# L = l-1 + attn(l-1) + MLP(attn(l-1) + l-1)  

In [1]:
"""
SD3 Causal Path Experiment
==========================
 
목표:
  같은 프롬프트 seed 100개에 대해 각 JointTransformerBlock을 분해:
 
    L = L_minus_1 + delta_attn + delta_mlp
      = L_minus_1
        + (L_after_attn - L_minus_1)   ← attention 기여분
        + (L_output - L_after_attn)    ← MLP 기여분
 
  블록별로 다른 프롬프트들이 같은 방향의 변화를 만드는지 (causal path 유사도) 분석.
 
Hook 위치:
  ① block 입력 시점        → L_minus_1    (hidden_states)
  ② norm2.forward_pre_hook → L_after_attn (attention residual 적용 후, MLP 이전)
  ③ block 출력 시점        → L_output     (block_output[1], image stream)
 
주의:
  - image tokens만 저장 (text tokens = out[0] 은 무시)
  - CFG batch=2 중 conditional 절반만 저장
  - f32 변환 후 뺄셈 (f16 rounding 오차 방지)
  - compact stats으로 저장 (full tensor 저장 시 메모리 ~90GB)
"""

'\nSD3 Causal Path Experiment\n==========================\n \n목표:\n  같은 프롬프트 seed 100개에 대해 각 JointTransformerBlock을 분해:\n \n    L = L_minus_1 + delta_attn + delta_mlp\n      = L_minus_1\n        + (L_after_attn - L_minus_1)   ← attention 기여분\n        + (L_output - L_after_attn)    ← MLP 기여분\n \n  블록별로 다른 프롬프트들이 같은 방향의 변화를 만드는지 (causal path 유사도) 분석.\n \nHook 위치:\n  ① block 입력 시점        → L_minus_1    (hidden_states)\n  ② norm2.forward_pre_hook → L_after_attn (attention residual 적용 후, MLP 이전)\n  ③ block 출력 시점        → L_output     (block_output[1], image stream)\n \n주의:\n  - image tokens만 저장 (text tokens = out[0] 은 무시)\n  - CFG batch=2 중 conditional 절반만 저장\n  - f32 변환 후 뺄셈 (f16 rounding 오차 방지)\n  - compact stats으로 저장 (full tensor 저장 시 메모리 ~90GB)\n'

In [ ]:
"""
SD3 Causal Path Experiment
==========================

sd3_hook.py의 공유 hook 로직을 사용.
verify와 동일한 on_capture 콜백 구조 사용.

목표:
  100개 프롬프트 × 24블록 × {uncond, cond} 에 대해
  delta_attn, delta_mlp를 추출하고 cross-prompt 방향 유사도 분석.
"""

import sys
import torch
import numpy as np
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from diffusers import StableDiffusion3Pipeline

# ── sd3_hook.py 위치 설정 ─────────────────────────────────────
# sd3_hook.py가 있는 디렉토리 경로를 여기에 입력
SD3_HOOK_DIR = "."   # ← 수정
sys.path.insert(0, SD3_HOOK_DIR)

from sd3_hook import patch_transformer, attach_step_tracker

# ──────────────────────────────────────────────────────────────
DEVICE              = "cuda"
DTYPE               = torch.float16
MODEL_ID            = "stabilityai/stable-diffusion-3-medium-diffusers"
OUTPUT_DIR          = Path("/data8/haeun/diffusion2/sd3_causal_path")
NUM_INFERENCE_STEPS = 28
CAPTURE_STEPS       = {0}   # {0}: step 0만, {1}: step 1만, {0,1}: 둘 다

# 프롬프트: 2개
PROMPTS = ["an apple"]   # 프롬프트 1개

# 시드: 0, 5, 10, ..., 495  →  100개
SEEDS = list(range(0, 500, 5))

assert len(SEEDS) == 100

# 실험 단위: 1개 프롬프트 × 100개 시드 = 100회 실행
# store.data[block_idx][n] 에서 n = seed_idx (0~99)
RUNS = [(PROMPTS[0], seed) for seed in SEEDS]
print(f"총 실험 수: {len(RUNS)}  (prompt='{PROMPTS[0]}', {len(SEEDS)} seeds)")

# ──────────────────────────────────────────────────────────────
print("Loading SD3 pipeline...")
pipe = StableDiffusion3Pipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

NUM_BLOCKS = len(pipe.transformer.transformer_blocks)
print(f"✓ {NUM_BLOCKS}개 JointTransformerBlock")

# ──────────────────────────────────────────────────────────────
# Activation Store
# ──────────────────────────────────────────────────────────────
class ActivationStore:
    def __init__(self):
        self.enabled = False
        # data[block_idx] = [
        #   { "uncond": compact_stats, "cond": compact_stats, "recon": ... },
        #   ...  (프롬프트 수만큼)
        # ]
        self.data: dict[int, list] = defaultdict(list)

    def enable(self):  self.enabled = True
    def disable(self): self.enabled = False

    def record(self,
               block_idx:     int,
               L_minus_1:     torch.Tensor,   # (batch, 4096, 1536), f16, CPU
               L_after_attn:  torch.Tensor,   # (batch, 4096, 1536), f16, CPU
               L_output:      torch.Tensor,   # (batch, 4096, 1536), f16, CPU
               batch_size:    int,
               ):
        if not self.enabled:
            return

        # ★ f32 변환 먼저, 뺄셈 나중 (f16 rounding 오차 방지)
        lm1_f32 = L_minus_1.float()    # (batch, 4096, 1536)
        lat_f32 = L_after_attn.float() # (batch, 4096, 1536)
        lo_f32  = L_output.float()     # (batch, 4096, 1536)

        delta_attn = lat_f32 - lm1_f32   # attention 기여분
        delta_mlp  = lo_f32  - lat_f32   # MLP 기여분

        def _to_compact_stats(t: torch.Tensor) -> dict:
            """
            (4096, 1536) → compact stats
            full tensor 저장 시 ~90GB → compact stats으로 수백 MB로 절약

            frob:      Frobenius norm (scalar)
            tok_norms: per-token L2 norm (4096,) f32  — 공간 패턴 분석용
            chan_mean: channel-wise mean (1536,) f32  — cross-prompt 방향 유사도용
            """
            return {
                "frob":      float(t.norm()),
                "tok_norms": t.norm(dim=-1).cpu().numpy().astype(np.float32),  # (4096,)
                "chan_mean": t.mean(dim=0).cpu().numpy().astype(np.float32),   # (1536,)
            }

        entry = {}
        for i, label in enumerate(["uncond", "cond"]):
            if i >= batch_size:
                break   # batch_size=1이면 uncond만
            entry[label] = {
                "lm1": _to_compact_stats(lm1_f32[i]),
                "da":  _to_compact_stats(delta_attn[i]),
                "dm":  _to_compact_stats(delta_mlp[i]),
            }

        # recon 오차 (cond 기준으로 기록)
        if batch_size >= 2:
            recon_err = (lm1_f32[1] + delta_attn[1] + delta_mlp[1] - lo_f32[1]).abs().sum().item()
            recon_rel = recon_err / (lo_f32[1].abs().sum().item() + 1e-8)
        else:
            recon_err = (lm1_f32[0] + delta_attn[0] + delta_mlp[0] - lo_f32[0]).abs().sum().item()
            recon_rel = recon_err / (lo_f32[0].abs().sum().item() + 1e-8)

        entry["recon_err_sum"] = recon_err
        entry["recon_rel_sum"] = recon_rel

        self.data[block_idx].append(entry)

    @property
    def n_prompts(self):
        return len(next(iter(self.data.values()))) if self.data else 0


store = ActivationStore()

# ── on_capture 콜백 ───────────────────────────────────────────
# verify.py의 on_capture와 동일한 구조
def on_capture(block_idx, L_minus_1, L_after_attn, L_output, batch_size):
    store.record(
        block_idx    = block_idx,
        L_minus_1    = L_minus_1,
        L_after_attn = L_after_attn,
        L_output     = L_output,
        batch_size   = batch_size,
    )

# ── 패치 ──────────────────────────────────────────────────────
already_patched = patch_transformer(pipe.transformer, on_capture)

if already_patched > 0:
    print(f"⚠ {already_patched}/{NUM_BLOCKS}개 블록 이미 패치됨 — 스킵")
else:
    print(f"✓ {NUM_BLOCKS}개 블록 패치 완료")

# ── Step tracker ──────────────────────────────────────────────
step_counter = attach_step_tracker(
    pipe.transformer,
    on_step=lambda step_idx: step_idx in CAPTURE_STEPS
)

if step_counter is None:
    print("⚠ Step tracker 이미 부착됨")
    # store.enabled은 on_capture 내부에서 확인하므로
    # step tracker의 hook_active 대신 store.enabled로 제어
else:
    print("✓ Step tracker 부착 완료")

# store.enabled를 step tracker hook_active와 동기화
_original_on_step = lambda step_idx: step_idx in CAPTURE_STEPS

def _synced_on_capture(block_idx, L_minus_1, L_after_attn, L_output, batch_size):
    # store.enabled가 True일 때만 record
    store.record(
        block_idx=block_idx,
        L_minus_1=L_minus_1,
        L_after_attn=L_after_attn,
        L_output=L_output,
        batch_size=batch_size,
    )

# step tracker가 hook_active를 제어하므로 store.enabled는 별도 관리
# on_capture는 항상 호출되지만, store.enabled로 저장 여부 결정
if step_counter is not None:
    orig_on_step_tracked = pipe.transformer.forward

    import functools
    @functools.wraps(orig_on_step_tracked)
    def _step_with_store(*args, **kwargs):
        # step counter 기반으로 store enable/disable
        store.enabled = (step_counter[0] in CAPTURE_STEPS)
        result = orig_on_step_tracked(*args, **kwargs)
        return result

    pipe.transformer.forward = _step_with_store

# ──────────────────────────────────────────────────────────────
# 추출 루프
# (prompt, seed) 조합 200회 실행
# store에는 순서대로 쌓임:
#   n=0~99   → "an apple",  seed 0,5,...,495
#   n=100~199→ "a cat",     seed 0,5,...,495
# ──────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"\n▶ 추출 시작: {len(RUNS)}회 실행 (step {CAPTURE_STEPS})\n")

for run_idx, (prompt, seed) in enumerate(tqdm(RUNS, desc="Extracting")):
    if step_counter is not None:
        step_counter[0] = 0   # 매 실행마다 denoising step counter 리셋
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    with torch.no_grad():
        _ = pipe(
            prompt=prompt,
            num_inference_steps=NUM_INFERENCE_STEPS,
            generator=generator,
        )

store.disable()
print(f"\n✓ 캡처 완료: {store.n_prompts}회 실행 × {NUM_BLOCKS}개 블록")
print(f"   = {len(PROMPTS)}개 프롬프트 × {len(SEEDS)}개 시드")

/home/haeun/miniconda3/envs/C3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


총 실험 수: 100  (prompt='an apple', 100 seeds)
Loading SD3 pipeline...


Loading pipeline components...: 100%|██████████| 9/9 [00:02<00:00,  4.49it/s]


✓ 24개 JointTransformerBlock
✓ 24개 블록 패치 완료
✓ Step tracker 부착 완료

▶ 추출 시작: 100회 실행 (step {0})



Extracting: 100%|██████████| 100/100 [45:00<00:00, 27.01s/it]



✓ 캡처 완료: 100회 실행 × 24개 블록
   = 1개 프롬프트 × 100개 시드


OSError: [Errno 28] No space left on device

In [2]:
import pickle
from pathlib import Path

# 용량 있는 경로로 설정
NEW_OUTPUT_DIR = Path("/data8/haeun/diffusion2/sd3_causal_path")   # ← 수정
NEW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

save_path = NEW_OUTPUT_DIR / "activations.pkl"
with open(save_path, "wb") as f:
    pickle.dump({
        "data": dict(store.data),
        "meta": {
            "prompts":       PROMPTS,
            "seeds":         SEEDS,
            "runs":          RUNS,
            "capture_steps": list(CAPTURE_STEPS),
            "num_blocks":    NUM_BLOCKS,
        }
    }, f)
print(f"✓ 저장 완료 → {save_path}  ({save_path.stat().st_size / 1e6:.1f} MB)")

✓ 저장 완료 → /data8/haeun/diffusion2/sd3_causal_path/activations.pkl  (325.8 MB)


In [6]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── 저장 경로 설정 (용량 있는 경로로 수정) ───────────────────
NEW_OUTPUT_DIR = Path("/data8/haeun/diffusion2/sd3_causal_path")   # ← 수정
NEW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 저장 ──────────────────────────────────────────────────────
save_path = NEW_OUTPUT_DIR / "activations.pkl"
with open(save_path, "wb") as f:
    pickle.dump({
        "data": dict(store.data),
        "meta": {
            "prompts":       PROMPTS,
            "seeds":         SEEDS,
            "runs":          RUNS,
            "capture_steps": list(CAPTURE_STEPS),
            "num_blocks":    NUM_BLOCKS,
        }
    }, f)
print(f"✓ 저장 완료 → {save_path}  ({save_path.stat().st_size / 1e6:.1f} MB)")

# ──────────────────────────────────────────────────────────────
# 분석 설정
# ──────────────────────────────────────────────────────────────
data   = store.data
N      = store.n_prompts   # 100 (시드 수)
B      = NUM_BLOCKS        # 24
PROMPT = PROMPTS[0]        # "an apple"
OUTPUT_DIR = NEW_OUTPUT_DIR

# ──────────────────────────────────────────────────────────────
# 유틸 함수
# ──────────────────────────────────────────────────────────────
def cosine_sim_matrix(vecs: np.ndarray) -> np.ndarray:
    """(N, D) → (N, N) pairwise cosine similarity"""
    norms  = np.linalg.norm(vecs, axis=1, keepdims=True)
    normed = vecs / np.clip(norms, 1e-8, None)
    return (normed @ normed.T).astype(np.float32)

def mean_off_diagonal(sim: np.ndarray) -> float:
    mask = ~np.eye(len(sim), dtype=bool)
    return float(sim[mask].mean())

def collect_stat(key: str, stat: str, cfg: str) -> np.ndarray:
    """data[block][seed][cfg][key][stat] → (N, B)"""
    return np.array([[data[b][n][cfg][key][stat] for b in range(B)] for n in range(N)])

# ──────────────────────────────────────────────────────────────
# Pooling 함수
# ──────────────────────────────────────────────────────────────
def get_mean_pooled(key: str, cfg: str) -> np.ndarray:
    """mean pooling: chan_mean (1536,) 그대로 사용. return: (N, B, 1536)"""
    return np.array([
        [data[b][n][cfg][key]["chan_mean"] for b in range(B)]
        for n in range(N)
    ])

def get_top10_pooled(key: str, cfg: str) -> np.ndarray:
    """
    top-10% token pooling 근사.
    compact stats만으로는 정확한 구현 불가 → tok_norms 스케일 가중치 사용.
    return: (N, B, 1536)
    """
    TOP_K_RATIO = 0.1
    result = np.zeros((N, B, 1536), dtype=np.float32)
    for n in range(N):
        for b in range(B):
            tok_norms      = data[b][n][cfg][key]["tok_norms"]  # (4096,)
            chan_mean       = data[b][n][cfg][key]["chan_mean"]  # (1536,)
            k               = max(1, int(len(tok_norms) * TOP_K_RATIO))
            top_idx         = np.argpartition(tok_norms, -k)[-k:]
            top_norms       = tok_norms[top_idx]
            weighted_scale  = top_norms.mean() / (tok_norms.mean() + 1e-8)
            result[n, b]    = chan_mean * weighted_scale
    return result

# ──────────────────────────────────────────────────────────────
# CFG별 분석 루프: cond → OUTPUT_DIR/cond/, uncond → OUTPUT_DIR/uncond/
# ──────────────────────────────────────────────────────────────
for CFG in ["cond", "uncond"]:
    cfg_dir = OUTPUT_DIR / CFG
    cfg_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*60}")
    print(f"  분석 시작: [{CFG}]  →  {cfg_dir}")
    print(f"{'='*60}")

    # ── Fig 1: Frobenius norm ──────────────────────────────────
    frob_lm1 = collect_stat("lm1", "frob", CFG)
    frob_da  = collect_stat("da",  "frob", CFG)
    frob_dm  = collect_stat("dm",  "frob", CFG)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, mat, label, color in zip(
        axes,
        [frob_lm1, frob_da, frob_dm],
        ["||L_minus_1||_F", "||delta_attn||_F", "||delta_mlp||_F"],
        ["steelblue", "darkorange", "seagreen"],
    ):
        mu, sd = mat.mean(0), mat.std(0)
        ax.plot(mu, "o-", color=color, markersize=4)
        ax.fill_between(range(B), mu - sd, mu + sd, alpha=0.2, color=color)
        ax.set_xlabel("Block index"); ax.set_ylabel("Frobenius norm")
        ax.set_title(label, fontweight="bold")
        ax.grid(True, alpha=0.35); ax.set_xticks(range(0, B, 2))
    plt.suptitle(f'"{PROMPT}" [{CFG}] — Residual norms  (mean ± σ,  {N} seeds)')
    plt.tight_layout()
    plt.savefig(cfg_dir / "fig1_frob_norms.png", dpi=150); plt.close()
    print("Saved: fig1_frob_norms.png")

    # ── Fig 2: Attention vs MLP 기여 비율 ─────────────────────
    ratio = frob_da / np.clip(frob_dm, 1e-8, None)
    fig, ax = plt.subplots(figsize=(10, 4))
    mu, sd = ratio.mean(0), ratio.std(0)
    ax.plot(mu, "o-", color="darkorange", markersize=4)
    ax.fill_between(range(B), mu - sd, mu + sd, alpha=0.2, color="darkorange")
    ax.axhline(1.0, color="red", linestyle="--", linewidth=1, label="attn = mlp")
    ax.set_xlabel("Block index"); ax.set_ylabel("||delta_attn|| / ||delta_mlp||")
    ax.set_title(f'"{PROMPT}" [{CFG}] — Attention vs MLP ratio', fontweight="bold")
    ax.legend(); ax.grid(True, alpha=0.35); ax.set_xticks(range(0, B, 2))
    plt.tight_layout()
    plt.savefig(cfg_dir / "fig2_attn_mlp_ratio.png", dpi=150); plt.close()
    print("Saved: fig2_attn_mlp_ratio.png")

    # ── Fig 3~5: pooling 방법별 ────────────────────────────────
    POOLING_METHODS = {
        "mean":  get_mean_pooled,
        "top10": get_top10_pooled,
    }

    for pool_name, pool_fn in POOLING_METHODS.items():
        print(f"\n  ── Pooling: {pool_name} ──")

        paths = {
            "da": pool_fn("da", CFG),   # (N, B, 1536)
            "dm": pool_fn("dm", CFG),   # (N, B, 1536)
        }

        # Fig 3: 블록별 cross-seed cosine similarity (mean ± std)
        fig, ax = plt.subplots(figsize=(10, 4))
        for key, color, marker, label in [
            ("da", "darkorange", "o", "delta_attn"),
            ("dm", "seagreen",   "s", "delta_mlp"),
        ]:
            sim_mean = np.zeros(B)
            sim_std  = np.zeros(B)
            for b in range(B):
                vecs        = paths[key][:, b, :]
                sim         = cosine_sim_matrix(vecs)
                off         = sim[~np.eye(N, dtype=bool)]
                sim_mean[b] = off.mean()
                sim_std[b]  = off.std()
            ax.plot(sim_mean, f"{marker}-", color=color, markersize=4, label=label)
            ax.fill_between(range(B), sim_mean - sim_std, sim_mean + sim_std,
                            alpha=0.15, color=color)

        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
        ax.set_xlabel("Block index")
        ax.set_ylabel("Cross-seed cosine sim  (mean ± std)")
        ax.set_title(
            f'"{PROMPT}" [{CFG}]  pooling={pool_name}\n'
            f"블록별 cross-seed similarity  ({N} seeds)",
            fontweight="bold"
        )
        ax.legend(); ax.set_ylim(-0.3, 1.0); ax.grid(True, alpha=0.35)
        ax.set_xticks(range(0, B, 2))
        plt.tight_layout()
        fname = f"fig3_{pool_name}_block_sim.png"
        plt.savefig(cfg_dir / fname, dpi=150); plt.close()
        print(f"  Saved: {fname}")

        # Fig 4: (100×100) seed-pair path similarity heatmap
        for key, label in [("da", "delta_attn"), ("dm", "delta_mlp")]:
            path_sim = np.zeros((N, N), dtype=np.float32)
            for b in range(B):
                path_sim += cosine_sim_matrix(paths[key][:, b, :])
            path_sim /= B

            fig, ax = plt.subplots(figsize=(8, 7))
            im = ax.imshow(path_sim, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
            plt.colorbar(im, ax=ax)
            ax.set_xlabel("Seed index"); ax.set_ylabel("Seed index")
            ax.set_title(
                f'"{PROMPT}" [{CFG}]  {label}  pooling={pool_name}\n'
                f"Seed-pair path similarity  (24블록 평균 cosine sim)",
                fontweight="bold"
            )
            plt.tight_layout()
            fname = f"fig4_{pool_name}_{key}_seed_heatmap.png"
            plt.savefig(cfg_dir / fname, dpi=150); plt.close()
            print(f"  Saved: {fname}")

        # Fig 5: (24×24) block-pair cosine similarity 행렬
        for key, label in [("da", "delta_attn"), ("dm", "delta_mlp")]:
            mean_vecs = paths[key].mean(axis=0)        # (B, 1536)
            block_sim = cosine_sim_matrix(mean_vecs)   # (B, B)

            fig, ax = plt.subplots(figsize=(8, 7))
            im = ax.imshow(block_sim, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
            plt.colorbar(im, ax=ax)
            ax.set_xlabel("Block index"); ax.set_ylabel("Block index")
            ax.set_xticks(range(0, B, 2)); ax.set_yticks(range(0, B, 2))
            ax.set_title(
                f'"{PROMPT}" [{CFG}]  {label}  pooling={pool_name}\n'
                f"Block-pair cosine similarity  (시드 평균 벡터 기준)",
                fontweight="bold"
            )
            plt.tight_layout()
            fname = f"fig5_{pool_name}_{key}_block_heatmap.png"
            plt.savefig(cfg_dir / fname, dpi=150); plt.close()
            print(f"  Saved: {fname}")

    # ── Fig 6: 공간 패턴 heatmap ───────────────────────────────
    for key, label in [("da", "delta_attn"), ("dm", "delta_mlp")]:
        ncols = 6; nrows = (B + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3 * nrows))
        for b in range(B):
            mean_tok = np.stack([data[b][n][CFG][key]["tok_norms"] for n in range(N)]).mean(0)
            hw = int(mean_tok.shape[0] ** 0.5)
            ax = axes.flatten()[b]
            if hw * hw == mean_tok.shape[0]:
                ax.imshow(mean_tok.reshape(hw, hw), cmap="hot", interpolation="nearest")
            else:
                ax.plot(mean_tok, linewidth=0.8)
            ax.set_title(f"Block {b}", fontsize=8); ax.axis("off")
        for ax in axes.flatten()[B:]:
            ax.set_visible(False)
        plt.suptitle(f'"{PROMPT}" [{CFG}] — ||{label}|| per block  (64×64)', fontweight="bold")
        plt.tight_layout()
        plt.savefig(cfg_dir / f"fig6_spatial_{key}.png", dpi=150); plt.close()
        print(f"Saved: fig6_spatial_{key}.png")

    # ── Recon 오차 ────────────────────────────────────────────
    recon_err_arr = np.array([[data[b][n]["recon_err_sum"] for b in range(B)] for n in range(N)])
    recon_rel_arr = np.array([[data[b][n]["recon_rel_sum"] for b in range(B)] for n in range(N)])

    print(f"\n{'='*60}")
    print(f"Reconstruction check [{CFG}]  (float32, 100 seeds 평균)")
    print(f"{'Blk':>4}  {'err_sum(mean)':>14}  {'rel_sum(mean)':>14}  판정")
    print("-" * 60)
    for b in range(B):
        r_mean  = recon_rel_arr[:, b].mean()
        verdict = "✓" if r_mean < 1e-5 else ("△" if r_mean < 1e-3 else "✗")
        print(f"  {b:>3}  {recon_err_arr[:,b].mean():>14.3e}  {r_mean:>14.8f}  {verdict}")
    print("=" * 60)

print(f"""
✓ 완료!

저장 구조:
  {OUTPUT_DIR}/
  ├── cond/
  │   ├── fig1_frob_norms.png
  │   ├── fig2_attn_mlp_ratio.png
  │   ├── fig3_mean_block_sim.png
  │   ├── fig3_top10_block_sim.png
  │   ├── fig4_mean_da/dm_seed_heatmap.png
  │   ├── fig4_top10_da/dm_seed_heatmap.png
  │   ├── fig5_mean_da/dm_block_heatmap.png
  │   ├── fig5_top10_da/dm_block_heatmap.png
  │   └── fig6_spatial_da/dm.png
  └── uncond/
      └── (동일 구조)
""")

✓ 저장 완료 → /data8/haeun/diffusion2/sd3_causal_path/activations.pkl  (325.8 MB)

  분석 시작: [cond]  →  /data8/haeun/diffusion2/sd3_causal_path/cond
Saved: fig1_frob_norms.png
Saved: fig2_attn_mlp_ratio.png

  ── Pooling: mean ──


/tmp/ipykernel_3422658/2501570117.py:169: UserWarning: Glyph 48660 (\N{HANGUL SYLLABLE BEUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:169: UserWarning: Glyph 47197 (\N{HANGUL SYLLABLE ROG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:169: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:171: UserWarning: Glyph 48660 (\N{HANGUL SYLLABLE BEUL}) missing from font(s) DejaVu Sans.
  plt.savefig(cfg_dir / fname, dpi=150); plt.close()
/tmp/ipykernel_3422658/2501570117.py:171: UserWarning: Glyph 47197 (\N{HANGUL SYLLABLE ROG}) missing from font(s) DejaVu Sans.
  plt.savefig(cfg_dir / fname, dpi=150); plt.close()
/tmp/ipykernel_3422658/2501570117.py:171: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from font(s) DejaVu Sans.
  plt.savefig(cfg_dir / fname, dpi=150); plt.close()


  Saved: fig3_mean_block_sim.png


/tmp/ipykernel_3422658/2501570117.py:190: UserWarning: Glyph 48660 (\N{HANGUL SYLLABLE BEUL}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:190: UserWarning: Glyph 47197 (\N{HANGUL SYLLABLE ROG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:190: UserWarning: Glyph 54217 (\N{HANGUL SYLLABLE PYEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:190: UserWarning: Glyph 44512 (\N{HANGUL SYLLABLE GYUN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:192: UserWarning: Glyph 48660 (\N{HANGUL SYLLABLE BEUL}) missing from font(s) DejaVu Sans.
  plt.savefig(cfg_dir / fname, dpi=150); plt.close()
/tmp/ipykernel_3422658/2501570117.py:192: UserWarning: Glyph 47197 (\N{HANGUL SYLLABLE ROG}) missing from font(s) DejaVu Sans.
  plt.savefig(cfg_dir / fname, dpi=150); plt.close()
/tmp/ipykernel_3422658/2501570117.py:192: 

  Saved: fig4_mean_da_seed_heatmap.png
  Saved: fig4_mean_dm_seed_heatmap.png


/tmp/ipykernel_3422658/2501570117.py:210: UserWarning: Glyph 49884 (\N{HANGUL SYLLABLE SI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:210: UserWarning: Glyph 46300 (\N{HANGUL SYLLABLE DEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:210: UserWarning: Glyph 54217 (\N{HANGUL SYLLABLE PYEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:210: UserWarning: Glyph 44512 (\N{HANGUL SYLLABLE GYUN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:210: UserWarning: Glyph 48289 (\N{HANGUL SYLLABLE BEG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:210: UserWarning: Glyph 53552 (\N{HANGUL SYLLABLE TEO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_3422658/2501570117.py:210: UserWarning: Glyph 44592 (\N{HANGUL SYLLABLE GI}) missing from font

  Saved: fig5_mean_da_block_heatmap.png
  Saved: fig5_mean_dm_block_heatmap.png

  ── Pooling: top10 ──
  Saved: fig3_top10_block_sim.png
  Saved: fig4_top10_da_seed_heatmap.png
  Saved: fig4_top10_dm_seed_heatmap.png
  Saved: fig5_top10_da_block_heatmap.png
  Saved: fig5_top10_dm_block_heatmap.png
Saved: fig6_spatial_da.png
Saved: fig6_spatial_dm.png

Reconstruction check [cond]  (float32, 100 seeds 평균)
 Blk   err_sum(mean)   rel_sum(mean)  판정
------------------------------------------------------------
    0       0.000e+00      0.00000000  ✓
    1       0.000e+00      0.00000000  ✓
    2       0.000e+00      0.00000000  ✓
    3       0.000e+00      0.00000000  ✓
    4       0.000e+00      0.00000000  ✓
    5       0.000e+00      0.00000000  ✓
    6       0.000e+00      0.00000000  ✓
    7       0.000e+00      0.00000000  ✓
    8       0.000e+00      0.00000000  ✓
    9       0.000e+00      0.00000000  ✓
   10       0.000e+00      0.00000000  ✓
   11       0.000e+00      0.00000000  